# Week 1, Part 01: Defining the Problem and Acquiring the Data

<sub>[Next: Data Exploration: Provenance, Shape, and Quality](week1_02_exploration.ipynb)</sub>

<sub>New here? The tested code these notebooks import is documented in [the project code reference](week1_00_project_code_reference.ipynb).</sub>

This is the first notebook of the project.

### Contents

1. [The problem: preterm birth and equity](#problem)
2. [The questions I want this project to answer](#questions)
3. [Where the data comes from](#source)
4. [Data provenance and governance](#governance)
5. [Acquisition Audit](#audit)
6. [Where this sits in the data-science lifecycle](#lifecycle)
7. [Scope for this week, and what comes next](#scope)

### Setup

A short, documented setup so the rest of the notebook reads cleanly. First the imports and display options. `scipy` is here because we run real hypothesis tests later, and we fix a random seed so every `df.sample()` below is reproducible.

In [1]:
# Setup. Week 1 is acquisition + exploration + EDA, so we load the RAW exploration table:
# the 17 pre-delivery birth-certificate fields we gathered, with their original values untouched.
# (No engineered features, no modeling columns yet. Those come in later weeks.) This table is built
# from the COMPLETE NBER public-use files for 2018-2024, restricted to singletons. Every number below
# is computed live from it. scipy is here for the hypothesis tests we run as we go, and we fix a seed
# so every df.sample() is reproducible.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
RNG = 7  # fixed seed so every df.sample() below is reproducible

Load the raw exploration table once. It holds the 17 pre-delivery certificate fields we gathered this week, built from the complete 2018-2024 public-use files (singletons only), plus readable label maps for the coded fields and the published national birth totals we will validate our counts against.

In [2]:
DATA = Path("..") / "data" / "interim" / "natality_exploration_2019_2024.parquet"
df = pd.read_parquet(DATA)  # ALL U.S. singleton births 2018-2024, raw pre-delivery fields only

# Human-readable label maps for the coded NCHS fields.
EDU = {1: "<8th grade", 2: "9-12, no diploma", 3: "HS grad/GED", 4: "Some college",
       5: "Associate", 6: "Bachelor", 7: "Master", 8: "Doctorate/Prof"}
PAY = {1: "Private", 2: "Medicaid", 3: "Self-pay", 4: "Indian Health Svc", 5: "CHAMPUS/TRICARE",
       6: "Other govt", 8: "Other"}
RACE_ORDER = ["NH White", "NH Asian", "Hispanic", "NH Multiracial", "NH NHOPI", "NH AIAN", "NH Black"]
# Published total U.S. births (NCHS final; 2024 provisional). Singletons are about 96.8% of births.
PUBLISHED_BIRTHS = {2018: 3791712, 2019: 3747540, 2020: 3613647, 2021: 3664292, 2022: 3667758,
                    2023: 3596017, 2024: 3600000}

A colorblind-safe palette (the Okabe-Ito set), reused on every figure so the series looks like one piece. We give the four largest race groups distinct colors and leave the rest grey to keep attention where the data are most reliable.

In [3]:
PALETTE = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
           "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#999999",
           "ink": "#1a1a2e"}
RACE_COLOR = {r: PALETTE["grey"] for r in RACE_ORDER}
RACE_COLOR.update({"NH White": PALETTE["blue"], "NH Black": PALETTE["red"],
                   "Hispanic": PALETTE["green"], "NH Asian": PALETTE["orange"]})

Matplotlib house style, plus a tiny `style()` helper so every static figure gets a takeaway-as-title and labelled axes without repeating boilerplate.

In [4]:
plt.rcParams.update({"figure.figsize": (8.6, 4.8), "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
                     "axes.titlepad": 10, "font.family": "DejaVu Sans"})

def style(ax, title, xlabel, ylabel):
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    return ax

Marquee charts also get an interactive Plotly companion, so `ptmpl()` gives those figures the same dark-text, low-clutter house style.

In [5]:
pio.renderers.default = "notebook_connected"  # small files; interactive in Jupyter / nbviewer

def ptmpl(fig, title, xtitle, ytitle, height=440):
    fig.update_layout(
        title=dict(text=title, x=0.02, font=dict(size=16, color=PALETTE["ink"])),
        xaxis_title=xtitle, yaxis_title=ytitle, plot_bgcolor="white", paper_bgcolor="white",
        font=dict(family="Inter, system-ui, sans-serif", size=13, color=PALETTE["ink"]),
        margin=dict(l=60, r=30, t=60, b=50), height=height,
        legend=dict(orientation="h", yanchor="bottom", y=1.03, x=0, font=dict(color=PALETTE["ink"])))
    fig.update_xaxes(showgrid=False, linecolor="#cccccc")
    fig.update_yaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    return fig

Finally, the 2 helpers we lean on most. `rate_by()` returns the preterm rate and count for any grouping column, and it is the workhorse of the entire EDA. `risk_ratio_ci()` goes 1 step further for binary flags: it returns the risk ratio (flag=1 vs flag=0) with a 95% log-normal confidence interval, so we can report an effect size and its uncertainty together, not just a p-value. Both are worth reading once.

In [6]:
def rate_by(frame, col, dropna_target=True):
    """Preterm rate (%) and count for any grouping column. The workhorse of the whole EDA."""
    d = frame.dropna(subset=["preterm"]) if dropna_target else frame
    g = d.groupby(col)["preterm"].agg(rate="mean", n="size")
    g["rate"] = g["rate"] * 100
    return g

def risk_ratio_ci(frame, flag_col, target="preterm"):
    """Risk ratio (flag=1 vs flag=0) with a 95% log-normal CI. Returns (rr, lo, hi, r1, r0)."""
    d = frame.dropna(subset=[target, flag_col])
    g = d.groupby(flag_col)[target].agg(["mean", "sum", "size"])
    r1, r0 = g.loc[1.0, "mean"], g.loc[0.0, "mean"]
    a, n1 = g.loc[1.0, "sum"], g.loc[1.0, "size"]
    c, n0 = g.loc[0.0, "sum"], g.loc[0.0, "size"]
    rr = r1 / r0
    se = np.sqrt(1/a - 1/n1 + 1/c - 1/n0)
    return rr, rr*np.exp(-1.96*se), rr*np.exp(1.96*se), r1*100, r0*100

### First, a look at the raw material

Each notebook stands on its own, so we start the way a real analysis session does: load the table and look at a handful of actual rows. One record is one singleton birth.

In [7]:
df.sample(5, random_state=RNG)

,birth_year,mager,meduc,bmi,cig_0,priorlive,priordead,married,rf_pdiab,rf_phype,rf_ppterm,rf_cesar,wic,first_trimester_care,pay,mother_race_ethnicity,preterm
19117181,2023,26,6.0,26.600000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,Hispanic,0.0
19115281,2023,36,7.0,24.000000,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,NH White,0.0
24082594,2024,28,4.0,23.200001,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,NH Multiracial,0.0
3195108,2018,30,3.0,25.700001,0.0,4.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,Hispanic,0.0
6895695,2019,30,6.0,19.000000,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,NH White,0.0


<a id="problem"></a>
## The problem: preterm birth and equity

**Preterm birth**, delivery before 37 completed weeks of gestation, is the leading cause of neonatal death and a major driver of lifelong neurodevelopmental and cardiometabolic disease. The [WHO](https://www.who.int/news-room/fact-sheets/detail/preterm-birth) sorts it into 3 sub-categories by how early the birth is: extremely preterm (before 28 weeks), very preterm (28 to under 32 weeks), and moderate-to-late preterm (32 to under 37 weeks). The scale is large: the WHO estimates about 13.4 million preterm births worldwide in 2020, and complications of prematurity are the leading cause of death in children under 5. Our target column collapses all of this into 1 flag (born before 37 weeks or not), which is the standard public-health definition and the one the certificate supports.

In the United States it is also one of the most stubborn racial inequities in all of public health. Non-Hispanic (NH) Black infants are born preterm far more often than NH White infants, and Black women die of pregnancy-related causes at [more than three times](https://www.kff.org/racial-equity-and-health-policy/racial-disparities-in-maternal-and-infant-health-current-status-and-key-issues/) the rate of White women. The part that has always struck is that this gap does not close with education or income, which is the first hint that the usual socioeconomic story is incomplete. A framing I hold to for the whole project: **race here is a social construct and a marker of exposure to racism and structural barriers, never a biological cause** of preterm birth, and nothing in these notebooks is a clinical tool. This is educational research, and it speaks in associational language, not causal claims.

One number to set expectations. The CDC/NCHS provisional 2024 figure ([VSRR 38](https://www.cdc.gov/nchs/data/vsrr/vsrr038.pdf)) puts the US preterm rate at 10.41%, but that is an *all-births* headline. This project studies *singletons* only. Multiples (twins, triplets, and more) are about 60% preterm and are excluded by design, because pooling them in would swamp the singleton signal we care about. So we expect our overall singleton rate to land meaningfully below 10.4%, and the EDA will check exactly that.

So this project has 2 motivations. First, can we predict preterm birth from information available *before* delivery? If we could, that would help clinicians target care. Second, and more importantly, how equitable is the outcome across race and ethnicity, and how much of the disparity can measured factors actually explain? We care about the second question more, because a prediction model that ignores the equity context would be worse than useless.

Sources: [WHO preterm-birth fact sheet](https://www.who.int/news-room/fact-sheets/detail/preterm-birth), [CDC/NCHS VSRR 38](https://www.cdc.gov/nchs/data/vsrr/vsrr038.pdf), [March of Dimes](https://www.marchofdimes.org/peristats/data), [KFF](https://www.kff.org/racial-equity-and-health-policy/racial-disparities-in-maternal-and-infant-health-current-status-and-key-issues/).

<a id="questions"></a>
## The questions we want this project to answer

These are the questions the whole project is organized around:

1. **How well can preterm birth be predicted from pre-delivery birth-certificate data?**
2. **How large is the racial disparity in preterm birth, and is it statistically real or just sampling noise?**
3. **Does the disparity shrink once we account for socioeconomic proxies like education?** This is the falsifiable version of the socioeconomic explanation hypothesis.
4. **Which recorded factors actually move the outcome, and by how much?**

This week we cannot answer the prediction question (that needs models, which is later work). But we can make real progress on questions 2 through 4 with careful EDA, and we can lay groundwork for question 1 by understanding exactly how much signal lives in these fields.

<a id="source"></a>
## Where the data comes from

The data are the **CDC/NCHS Natality public-use files**, distributed through the [NBER mirror](https://www.nber.org/research/data/vital-statistics-natality-birth-data). These are not a survey. They are a near-census assembled by the [National Vital Statistics System](https://www.cdc.gov/nchs/nvss/about_nvss.htm), in which NCHS contracts with the registration systems of the 50 states, DC, New York City, and five territories. Every birth produces a certificate; the clinical and demographic items come from hospital records and a maternal worksheet.

I settled on a **2018 to 2024** window. The short version: since 2016 every jurisdiction has used the [2003 revision](https://www.cdc.gov/nchs/nvss/revisions-of-the-us-standard-certificates-and-reports.htm) of the U.S. Standard Certificate of Live Birth, so every year in this window is on the same form with the same fields, which is what makes pooling them legitimate. A framing note I will carry throughout: the familiar 10.4% national preterm headline is an *all-births* figure, and this project studies *singletons* only (multiples are excluded by design because they are about 60% preterm and would swamp the signal), so we expect our overall rate to land meaningfully below 10.4%.

<a id="governance"></a>
## Data provenance and governance

Because so much rides on trusting these files, it is worth writing down where they come from and under what rules, before we compute a single statistic.

- **Near-census, not a survey.** The [National Vital Statistics System](https://www.cdc.gov/nchs/nvss/about_nvss.htm) registers essentially every birth in the country. NCHS works through the vital-registration systems of the 50 states, DC, New York City, and the territories, so coverage is meant to be the whole population rather than a sampled slice.
- **Where the items come from.** The demographic and clinical fields originate on the hospital worksheet and the facility birth record: some are transcribed from the medical chart, others are reported by the mother. Knowing which is which matters for the bias notes in notebook 02.
- **1 stable form across the window.** Every US jurisdiction had adopted the [2003 revision](https://www.cdc.gov/nchs/nvss/revisions-of-the-us-standard-certificates-and-reports.htm) of the birth certificate by 2016, so all of 2018 to 2024 sits on the same form with the same fields. That is what makes pooling the years legitimate rather than an apples-to-oranges mix.
- **Privacy and ethics.** The public-use files are de-identified and, as public secondary data, the analysis is IRB-exempt. There is nothing in them that identifies an individual mother or infant.
- **No geography post-2005.** For privacy the public-use files carry no state or county identifiers, so any state-level comparison later has to come from separate CDC WONDER aggregates rather than from these rows. That is noted as PLANNED work for a later week, not something this week can do.

The practical upshot for this notebook: because the source is a near-census on a single stable form, our coverage should land near 100% of singleton births each year, which is exactly the acquisition audit below.

<a id="audit"></a>
## Acquisition audit

Even with the complete files, we owe a check that what landed on disk matches the national totals, because a quietly truncated download or a botched singleton filter would show up here. Let me compare our singleton counts per year against the published births.

In [8]:
# Our singleton count per year vs the published national totals (singletons ~96.8% of births).
yr = df.groupby('birth_year').size()
rows = []
for y in sorted(PUBLISHED_BIRTHS):
    singletons = PUBLISHED_BIRTHS[y] * 0.968
    rows.append({'year': y, 'our_singletons': int(yr[y]), 'published_singletons': int(singletons),
                 'coverage_%': round(yr[y] / singletons * 100, 1)})
frac = pd.DataFrame(rows)
sing_total = sum(PUBLISHED_BIRTHS.values()) * 0.968
print(f'In total we hold {df.shape[0]:,} singleton rows, about '
      f'{df.shape[0]/sing_total*100:.1f}% of the ~{sing_total/1e6:.1f}M published singleton births in this window.')
frac

In total we hold 24,941,021 singleton rows, about 100.3% of the ~24.9M published singleton births in this window.


,year,our_singletons,published_singletons,coverage_%
0,2018,3674115,3670377,100.1
1,2019,3633587,3627618,100.2
2,2020,3504248,3498010,100.2
3,2021,3552604,3547034,100.2
4,2022,3558362,3550389,100.2
5,2023,3491735,3480944,100.3
6,2024,3526370,3484800,101.2


The exact numbers behind the chart, printed so nothing hides in the bars: our count, the approximate published singleton count, and the coverage ratio for each year.

In [9]:
print("Coverage of published singleton births, per year (values behind the chart):")
for _, r in frac.iterrows():
    print(f"  {int(r['year'])}: our {int(r['our_singletons']):>9,}  "
          f"vs published ~{int(r['published_singletons']):>9,}  = {r['coverage_%']:.1f}%")

Coverage of published singleton births, per year (values behind the chart):
  2018: our 3,674,115  vs published ~3,670,377  = 100.1%
  2019: our 3,633,587  vs published ~3,627,618  = 100.2%
  2020: our 3,504,248  vs published ~3,498,010  = 100.2%
  2021: our 3,552,604  vs published ~3,547,034  = 100.2%
  2022: our 3,558,362  vs published ~3,550,389  = 100.2%
  2023: our 3,491,735  vs published ~3,480,944  = 100.3%
  2024: our 3,526,370  vs published ~3,484,800  = 101.2%


Let me see that coverage as an interactive chart. With the complete files the bars should sit right around 100% every year (a touch over, because the published-singleton denominator is an approximation and 2024 is provisional). Hover any bar for its exact value.

In [10]:
fig = go.Figure(go.Bar(x=frac['year'].astype(str), y=frac['coverage_%'],
                       marker_color=PALETTE['sky'],
                       text=[f'{v:.1f}%' for v in frac['coverage_%']], textposition='outside',
                       hovertemplate='%{x}: %{y:.1f}% of published singletons<extra></extra>'))
fig.add_hline(y=100, line_dash='dot', line_color=PALETTE['grey'])
fig.update_yaxes(range=[0, max(105, frac['coverage_%'].max()*1.1)])
ptmpl(fig, 'Our singleton counts track the published totals every year (complete files)',
      'Birth year', 'Our count as % of published singletons')
fig.show()

**Reading the audit**

1. **We now have essentially the whole singleton population for each year, not a sample.** Coverage lands right around 100% every year (the table reads 100 to 101 percent, slightly over because the 96.8% singleton share is a round figure and 2024 births are still provisional, so the denominator is approximate). The point is that we hold essentially every singleton record; the only ones set aside are births whose obstetric gestational estimate is missing (null target) and the handful the singleton rule removes. There is no sampling step left to bias anything. We still cross-check representativeness the reproducible way in the EDA, by confirming the per-race rates reproduce the published NVSS ordering and magnitudes.
2. **Counts are comparable across years now**, because every year is the full population rather than an uneven sample. That said, I still default to **rates** for cross-year comparison, because rates are the right unit for a public-health question and are robust to the small year-to-year coverage differences from missing-target records.

<a id="lifecycle"></a>
## Where this sits in the data-science lifecycle

It helps to step back and place this week honestly on the full data-science lifecycle, so it is clear what this week covers versus what the coming weeks will build. The stages below are grouped: Covered this week, Planned (a later week, which the EDA will inform), or Out of scope for this project. This is a map of the plan, not a claim that the later stages are done.

| Lifecycle stage | Status | Where |
|---|---|---|
| Problem / business understanding | Covered this week | This notebook |
| Data governance and regulatory context | Covered this week | This notebook |
| Data acquisition and ingestion | Covered this week | This notebook |
| Data validation and schema checks | Covered this week | Notebook 02 |
| Data-quality assessment, cleaning, missing-value and outlier treatment | Covered this week | Notebook 02, 03 |
| EDA, descriptive stats, distribution and correlation analysis | Covered this week | Notebook 03 |
| Hypothesis testing (p-values, confidence intervals, effect sizes) | Covered this week | Notebook 03 |
| Visualization and storytelling | Covered this week | All week-1 notebooks and the dashboard |
| Feature engineering, encoding, interaction features | Planned | Week 2 |
| Leakage prevention and train / test / held-out splits | Planned | Week 2 |
| Calibration and threshold tuning | Planned | Week 3 |
| Supervised baselines and gradient boosting | Planned | Weeks 3 to 4 |
| Hyperparameter search (Optuna) | Planned | Week 4 |
| Evaluation metrics (AUC-ROC, AUC-PR, Brier, ECE / MCE) | Planned | Weeks 3 to 4 |
| First fairness evaluation (equal opportunity, demographic parity, disparate impact) with significance control | Planned | Week 4 |
| Deep-learning benchmark and interpretability | Planned | Later week |
| Ensemble / stacking and final model selection | Planned | Later week |
| Full fairness mitigation and tradeoff tables | Planned | Later week |
| SHAP interpretability and decomposition of the racial gap | Planned | Later week |
| Uncertainty / robustness and out-of-time validation | Planned | Later week |
| Model card and datasheet | Planned | Later week |
| Deployment (Streamlit) and monitoring | Planned | Later week |
| CDC WONDER aggregate state context and NHANES external sanity check | Planned | Later week |
| Streaming ingestion | Out of scope | Batch public-use files, no stream |
| A/B tests and RCTs | Out of scope | Observational data only |
| Row-level geographic joins | Out of scope | No geography in public-use files |
| RCT-grade causal inference | Out of scope | Associational study, not causal |

The Covered rows are this week's work: acquisition, exploration, quality auditing, EDA, and visualization. Everything from the feature-engineering stage on is Planned, so throughout this week it reads as intent, not as results, and the modeling and fairness rows will be filled in as the coming weeks build them. The Out-of-scope rows are honest boundaries: they are either impossible with these de-identified batch files (streaming, geography) or the wrong tool for an observational dataset (randomized experiments, causal inference).

<a id="scope"></a>
## Scope for this week, and what comes next

**This week** is acquisition, exploration, data-quality auditing, EDA, and visualization. That is the work in these three notebooks and the dashboard.

**What we plan to do in the coming weeks**, informed by what the EDA reveals, is roughly:

- **Feature engineering with a strict leakage audit.** Birth certificates record many fields *at* delivery (birthweight, Apgar score, labor interventions) that are near-perfect but useless predictors, because they do not exist when a prediction would actually be made. We will need a disciplined, tested gate that bars those, and we expect that to be the single most important engineering decision in the project.
- **Baseline models, calibration first.** Logistic regression and tree baselines, with honest attention to the fact that a rare outcome makes accuracy a meaningless metric.
- **A formal fairness audit.** Demographic parity, equal opportunity, calibration-by-group, with confidence intervals, to turn the disparity we expect to find this week into rigorous metrics.
- **Explainability and a decomposition of the gap**, to ask how much of the racial disparity measured factors can actually account for.

---

Author: [Parshv Patel](https://www.linkedin.com/in/parshv-patel-65a90326b/)  

--- 

Edit Logs:
